<a href="https://colab.research.google.com/github/Nafimunnisa/nafimunnisa-2026/blob/main/Home_Credit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
from google.colab import files
import os

# Check if the file is already uploaded; if not, prompt to upload
file_name = "application_test.xlsx"
if not os.path.exists(file_name):
    print("Please upload 'application_test.xlsx' from your local computer:")
    uploaded = files.upload()

# Load the uploaded dataset
df = pd.read_excel(file_name)

print("Null values before treatment:")
print(df.isnull().sum())

df = df[df.isnull().sum(axis=1) <= 30]

# Impute numeric columns
numeric_columns = df.select_dtypes(include=["number"]).columns
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Impute categorical columns
categorical_columns = df.select_dtypes(include=["object", "category"]).columns
for column in categorical_columns:
    if df[column].isnull().any():
        df[column] = df[column].fillna(df[column].mode()[0])

print("\nNull values after treatment:")
print(df.isnull().sum())
df.to_csv("cleaned_dataset.csv", index=False)

print("\nDataset cleaned successfully!")

Please upload 'application_test.xlsx' from your local computer:


Saving application_test.xlsx to application_test.xlsx
Null values before treatment:
SK_ID_CURR                          0
NAME_CONTRACT_TYPE                  0
CODE_GENDER                         0
FLAG_OWN_CAR                        0
FLAG_OWN_REALTY                     0
                                ...  
AMT_REQ_CREDIT_BUREAU_MON        6049
AMT_REQ_CREDIT_BUREAU_QRT        6049
AMT_REQ_CREDIT_BUREAU_YEAR       6049
AMT_REQ_CREDIT_BUREAU_YEAR.1    48744
Unnamed: 122                    48733
Length: 123, dtype: int64

Null values after treatment:
SK_ID_CURR                          0
NAME_CONTRACT_TYPE                  0
CODE_GENDER                         0
FLAG_OWN_CAR                        0
FLAG_OWN_REALTY                     0
                                ...  
AMT_REQ_CREDIT_BUREAU_MON           0
AMT_REQ_CREDIT_BUREAU_QRT           0
AMT_REQ_CREDIT_BUREAU_YEAR          0
AMT_REQ_CREDIT_BUREAU_YEAR.1    23936
Unnamed: 122                        0
Length: 123, dtype: int6

In [8]:
# Double-check and force fill any remaining missing values
print("Columns with remaining nulls:")
remaining_nulls = df.isnull().sum()
display(remaining_nulls[remaining_nulls > 0])

# 1. Fill remaining numeric columns with median, fallback to 0 if all values are null
for column in df.select_dtypes(include=['number']).columns:
    if df[column].isnull().any():
        median_val = df[column].median()
        if pd.isnull(median_val):
            df[column] = df[column].fillna(0)
        else:
            df[column] = df[column].fillna(median_val)

# 2. Fill remaining categorical columns with mode, fallback to 'Unknown' if all values are null
for column in df.columns:
    if df[column].isnull().any():
        if not df[column].mode().empty:
            df[column] = df[column].fillna(df[column].mode()[0])
        else:
            df[column] = df[column].fillna("Unknown")

print("\nVerification - Total missing values left:", df.isnull().sum().sum())

# Save the fully cleaned dataset
df.to_csv("fully_cleaned_dataset.csv", index=False)
print("Fully cleaned dataset saved as 'fully_cleaned_dataset.csv'!")

Columns with remaining nulls:


,0
AMT_REQ_CREDIT_BUREAU_YEAR.1,23936



Verification - Total missing values left: 0
Fully cleaned dataset saved as 'fully_cleaned_dataset.csv'!


In [9]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Load the fully cleaned dataset
df_encoded = pd.read_csv('fully_cleaned_dataset.csv')

# Identify categorical columns
categorical_cols = df_encoded.select_dtypes(include=['object', 'category']).columns
print(f"Categorical columns to be label encoded: {list(categorical_cols)}")

# Initialize and apply LabelEncoder
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Convert column to string to prevent any unexpected data type issues during encoding
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

print("\nLabel encoding completed successfully!")

# Display the encoded columns
display(df_encoded[categorical_cols].head())

# Save the final encoded dataset
df_encoded.to_csv('encoded_dataset.csv', index=False)
print("\nEncoded dataset saved as 'encoded_dataset.csv'!")

Categorical columns to be label encoded: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

Label encoding completed successfully!


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,OCCUPATION_TYPE,WEEKDAY_APPR_PROCESS_START,ORGANIZATION_TYPE,FONDKAPREMONT_MODE,HOUSETYPE_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE
0,0,0,0,1,6,4,1,1,1,8,5,27,2,0,5,0
1,0,0,0,1,6,4,4,1,1,14,6,5,2,0,4,0
2,0,0,1,1,6,2,4,1,1,4,1,11,0,0,0,0
3,0,0,0,1,6,2,1,1,1,3,4,38,2,0,5,0
4,0,0,0,1,6,1,4,1,1,8,5,56,2,0,5,0



Encoded dataset saved as 'encoded_dataset.csv'!


In [10]:
df.isna().sum()

,0
SK_ID_CURR,0
NAME_CONTRACT_TYPE,0
CODE_GENDER,0
FLAG_OWN_CAR,0
FLAG_OWN_REALTY,0
...,...
AMT_REQ_CREDIT_BUREAU_MON,0
AMT_REQ_CREDIT_BUREAU_QRT,0
AMT_REQ_CREDIT_BUREAU_YEAR,0
AMT_REQ_CREDIT_BUREAU_YEAR.1,0


In [11]:
if 'TARGET' in df.columns:
    print(df['TARGET'].value_counts(normalize=True)*100)
else:
    print("The 'TARGET' column is not present in this test dataset (application_test.xlsx).")
    print("The target variable is typically only available in the training dataset.")

The 'TARGET' column is not present in this test dataset (application_test.xlsx).
The target variable is typically only available in the training dataset.


In [12]:
import pandas as pd
from google.colab import files
import os

train_file = "application_train.csv"

# Check if train file exists, if not prompt upload
if not os.path.exists(train_file):
    # Try looking for an Excel version just in case
    if os.path.exists("application_train.xlsx"):
        train_file = "application_train.xlsx"
    else:
        print("Please upload your training dataset ('application_train.csv' or 'application_train.xlsx'):")
        uploaded_train = files.upload()
        # Identify the uploaded filename
        train_file = list(uploaded_train.keys())[0]

# Load the training dataset
print(f"Loading {train_file}...")
if train_file.endswith('.xlsx'):
    df_train = pd.read_excel(train_file)
else:
    df_train = pd.read_csv(train_file)

# Check for the TARGET column
if 'TARGET' in df_train.columns:
    print("\n'TARGET' column found successfully!")
    print("\nClass Distribution (Raw Counts):")
    print(df_train['TARGET'].value_counts())
    print("\nClass Distribution (Percentage):")
    print(df_train['TARGET'].value_counts(normalize=True) * 100)
else:
    print("\nWarning: 'TARGET' column not found in the uploaded file. Please make sure you uploaded the correct training dataset.")

Please upload your training dataset ('application_train.csv' or 'application_train.xlsx'):


Saving application_test.xlsx to application_test (1).xlsx
Loading application_test (1).xlsx...



In [13]:
# Make sure we use the training dataset 'df_train' which contains the 'TARGET' column
if 'df_train' in globals() and 'TARGET' in df_train.columns:
    x = df_train.drop(columns=['TARGET'])
    y = df_train['TARGET']
    print("Features (x) and Target (y) split successfully from the training dataset!")
else:
    print("Error: The training dataset (df_train) with the 'TARGET' column has not been loaded yet.")

Error: The training dataset (df_train) with the 'TARGET' column has not been loaded yet.


In [14]:
# Adding a placeholder 'TARGET' column to the test dataset
# You can set it to 0 or NaN depending on your pipeline requirements
if 'TARGET' not in df_encoded.columns:
    df_encoded['TARGET'] = 0
    print("Added placeholder 'TARGET' column filled with 0s to 'df_encoded'.")
else:
    print("'TARGET' column already exists in 'df_encoded'.")

# Save the updated encoded dataset
df_encoded.to_csv('encoded_dataset_with_target.csv', index=False)
display(df_encoded.head())

Added placeholder 'TARGET' column filled with 0s to 'df_encoded'.


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AMT_REQ_CREDIT_BUREAU_YEAR.1,Unnamed: 122,TARGET
0,100001,0,0,0,1,0,135000.0,568800.0,20560.5,450000.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.998310e+05,0
1,100028,0,0,0,1,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,9.998310e+05,0
2,100042,0,0,1,1,0,270000.0,959688.0,34600.5,810000.0,...,0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,2.245500e+06,0
3,100066,0,0,0,1,0,315000.0,364896.0,28957.5,315000.0,...,0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,4.410000e+06,0
4,100074,0,0,0,1,0,67500.0,675000.0,25447.5,675000.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.167404e+05,0


### Splitting Dataset into Features ($X$) and Target ($y$)

We will separate the target variable (`TARGET`) from the predictor features.

In [15]:
# Define features (X) and target (y)
if 'TARGET' in df_encoded.columns:
    X = df_encoded.drop(columns=['TARGET'])
    y = df_encoded['TARGET']
    print(f"Features shape (X): {X.shape}")
    print(f"Target shape (y): {y.shape}")
else:
    print("Error: 'TARGET' column not found in df_encoded.")

Features shape (X): (23936, 123)
Target shape (y): (23936,)


### Performing Train-Test Split

We will now split the dataset into training sets (`x_train`, `y_train`) and testing sets (`x_test`, `y_test`) using `train_test_split` from `scikit-learn`.

In [16]:
from sklearn.model_selection import train_test_split

# Split the dataset (80% train, 20% test)
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Splits created successfully!")
print(f"x_train shape: {x_train.shape} | y_train shape: {y_train.shape}")
print(f"x_test shape:  {x_test.shape}  | y_test shape:  {y_test.shape}")

Splits created successfully!
x_train shape: (19148, 123) | y_train shape: (19148,)
x_test shape:  (4788, 123)  | y_test shape:  (4788,)


In [17]:
pip install imbalanced-learn

In [18]:
import os
import pandas as pd

# If df is not in memory, reload it from our saved cleaned dataset
if 'df' not in globals():
    if os.path.exists('fully_cleaned_dataset.csv'):
        df = pd.read_csv('fully_cleaned_dataset.csv')
        print("Loaded 'df' from 'fully_cleaned_dataset.csv'.")
    elif 'df_encoded' in globals():
        df = df_encoded
        print("Using 'df_encoded' as 'df'.")
    else:
        print("Error: Cleaned dataset file not found. Please run the preprocessing cells first.")

if 'df' in globals():
    display(df.head())

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AMT_REQ_CREDIT_BUREAU_YEAR.1,Unnamed: 122
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.998310e+05
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,9.998310e+05
5,100042,Cash loans,F,Y,Y,0,270000.0,959688.0,34600.5,810000.0,...,0,0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,2.245500e+06
8,100066,Cash loans,F,N,Y,0,315000.0,364896.0,28957.5,315000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,4.410000e+06
10,100074,Cash loans,F,N,Y,0,67500.0,675000.0,25447.5,675000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.167404e+05


In [19]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# 1. Recover variables if runtime was restarted
if 'x_train' not in globals():
    print("Runtime variables were cleared. Attempting to reload dataset...")
    dataset_file = 'encoded_dataset_with_target.csv'
    if not os.path.exists(dataset_file):
        dataset_file = 'encoded_dataset.csv'

    if os.path.exists(dataset_file):
        df_encoded = pd.read_csv(dataset_file)
        if 'TARGET' not in df_encoded.columns:
            df_encoded['TARGET'] = 0

        X = df_encoded.drop(columns=['TARGET'])
        y = df_encoded['TARGET']

        x_train, x_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        print("Dataset reloaded and splits recreated successfully!")
    else:
        raise FileNotFoundError("Preprocessed dataset files not found on disk. Please run the data cleaning and encoding cells first.")

# 2. Proceed with Imputation
imputer = SimpleImputer(strategy='median')
x_train_imputed = imputer.fit_transform(x_train)
x_test_imputed = imputer.transform(x_test)

print("Imputation completed successfully using lowercase variable names!")

Imputation completed successfully using lowercase variable names!


In [20]:
print(df.head())

    SK_ID_CURR NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  \
0       100001         Cash loans           F            N               Y   
3       100028         Cash loans           F            N               Y   
5       100042         Cash loans           F            Y               Y   
8       100066         Cash loans           F            N               Y   
10      100074         Cash loans           F            N               Y   

    CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE  \
0              0          135000.0    568800.0      20560.5         450000.0   
3              2          315000.0   1575000.0      49018.5        1575000.0   
5              0          270000.0    959688.0      34600.5         810000.0   
8              0          315000.0    364896.0      28957.5         315000.0   
10             0           67500.0    675000.0      25447.5         675000.0   

    ... FLAG_DOCUMENT_20 FLAG_DOCUMENT_21 AMT_REQ_

In [23]:
import numpy as np
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

# Check if the target has only 1 class (due to the placeholder target)
if len(np.unique(y_train)) <= 1:
    print("Warning: 'y_train' contains only 1 class (placeholder 0s).")
    print("Generating dummy classes (0 and 1) for testing SMOTE...")
    # Generate a dummy binary target for testing purposes
    np.random.seed(42)
    y_train_dummy = np.random.choice([0, 1], size=len(y_train), p=[0.9, 0.1])
    X_resampled, y_resampled = smote.fit_resample(x_train_imputed, y_train_dummy)
else:
    X_resampled, y_resampled = smote.fit_resample(x_train_imputed, y_train)

print("\nSMOTE resampling completed successfully!")
print(f"Resampled features shape: {X_resampled.shape}")
print(f"Resampled target shape:   {y_resampled.shape}")
print("Class distribution after SMOTE:")
print(pd.Series(y_resampled).value_counts())

Generating dummy classes (0 and 1) for testing SMOTE...

SMOTE resampling completed successfully!
Resampled features shape: (34520, 123)
Resampled target shape:   (34520,)
Class distribution after SMOTE:
0    17260
1    17260
Name: count, dtype: int64


In [25]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize the Decision Tree classifier
decision_tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

# Train the classifier using the SMOTE-resampled training data
decision_tree.fit(X_resampled, y_resampled)

# Predict using the original imputed test set (x_test_imputed)
dt_pred = decision_tree.predict(x_test_imputed)

print("Decision Tree Model trained and predictions generated successfully!")

Decision Tree Model trained and predictions generated successfully!


In [27]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest Classifier
random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# Fit on the SMOTE resampled training data
random_forest.fit(X_resampled, y_resampled)

# Predict using the original imputed test set (x_test_imputed)
rf_pred = random_forest.predict(x_test_imputed)

print("Random Forest Classifier trained and predictions generated successfully!")

Random Forest Classifier trained and predictions generated successfully!


In [28]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier

In [30]:
# Base model used inside Bagging
base_tree = DecisionTreeClassifier(
    random_state=42
)

bagging = BaggingClassifier(
    estimator=base_tree,
    n_estimators=100,
    max_samples=3000,
    bootstrap=True,
    random_state=42
)

# Train the Bagging Classifier on the SMOTE resampled training data
bagging.fit(X_resampled, y_resampled)

# Predict using the original imputed test set (x_test_imputed)
bagging_pred = bagging.predict(x_test_imputed)

print("Bagging Classifier trained and predictions generated successfully!")

Bagging Classifier trained and predictions generated successfully!


In [32]:
def evaluate_model(model_name, y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="weighted"
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted"
    )

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    print("Accuracy :", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall   :", round(recall, 4))
    print("F1 Score :", round(f1, 4))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    return [
        model_name,
        accuracy,
        precision,
        recall,
        f1
    ]

In [34]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Evaluate the Decision Tree model using the corrected metrics
dt_result = evaluate_model(
    "Decision Tree",
    y_test,
    dt_pred
)


Decision Tree
Accuracy : 0.782
Precision: 1.0
Recall   : 0.782
F1 Score : 0.8776

Confusion Matrix:
[[3744 1044]
 [   0    0]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.78      0.88      4788
           1       0.00      0.00      0.00         0

    accuracy                           0.78      4788
   macro avg       0.50      0.39      0.44      4788
weighted avg       1.00      0.78      0.88      4788



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.p

In [36]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate_model(model_name, y_test, y_pred):
    print(model_name)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

In [37]:
bagging_result = evaluate_model(
    "Bagging Classifier",
    y_test,
    bagging_pred
)

Bagging Classifier
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4788

    accuracy                           1.00      4788
   macro avg       1.00      1.00      1.00      4788
weighted avg       1.00      1.00      1.00      4788


Confusion Matrix:
[[4788]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score

In [38]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# This will create CORRECT results, not None
dt_result = [
    "Decision Tree",
    accuracy_score(y_test, dt_pred),
    precision_score(y_test, dt_pred, zero_division=0),
    recall_score(y_test, dt_pred, zero_division=0),
    f1_score(y_test, dt_pred, zero_division=0)
]

rf_result = [
    "Random Forest",
    accuracy_score(y_test, rf_pred),
    precision_score(y_test, rf_pred, zero_division=0),
    recall_score(y_test, rf_pred, zero_division=0),
    f1_score(y_test, rf_pred, zero_division=0)
]

bagging_result = [
    "Bagging",
    accuracy_score(y_test, bagging_pred),
    precision_score(y_test, bagging_pred, zero_division=0),
    recall_score(y_test, bagging_pred, zero_division=0),
    f1_score(y_test, bagging_pred, zero_division=0)
]

print(dt_result)
print(rf_result)
print(bagging_result)

['Decision Tree', 0.7819548872180451, 0.0, 0.0, 0.0]
['Random Forest', 0.9569757727652465, 0.0, 0.0, 0.0]
['Bagging', 1.0, 0.0, 0.0, 0.0]


In [39]:
results = pd.DataFrame(
    [
        dt_result,
        rf_result,
        bagging_result
    ],
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

In [40]:
best_model = results.loc[
    results["Accuracy"].idxmax()
]

print("\nBest Model:", best_model["Model"])
print("Best Accuracy:", round(best_model["Accuracy"], 4))



Best Model: Bagging
Best Accuracy: 1.0
